In [5]:

%pip install azure-ai-projects
%pip install azure-identity
%pip install azure-ai-agents
%pip install jsonref
%pip install dotenv

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [dotenv]
Note: you may need to restart the kernel to use updated packages.


## Import Variables

In [6]:

import os, time, sys
import jsonref
from azure.ai.projects import AIProjectClient
from azure.identity import DefaultAzureCredential
from dotenv import load_dotenv
from azure.ai.agents.models import OpenApiTool, OpenApiAnonymousAuthDetails
from azure.ai.agents.models import ConnectedAgentTool, ListSortOrder, MessageRole, RunStepToolCallDetails, FunctionTool, RequiredFunctionToolCall, SubmitToolOutputsAction, ToolOutput
import functions


load_dotenv("globalparameters.env")

# Azure Search
AZURE_SEARCH_ENDPOINT = os.getenv("AZURE_SEARCH_ENDPOINT")
AZURE_SEARCH_API_KEY = os.getenv("AZURE_SEARCH_API_KEY")
INDEX_NAME = os.getenv("INDEX_NAME")
SKILLSET_NAME = os.getenv("SKILLSET_NAME")

# Azure OpenAI
AZURE_OPENAI_ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT")
AZURE_OPENAI_KEY = os.getenv("AZURE_OPENAI_KEY")
AZURE_OPENAI_VERSION = os.getenv("AZURE_OPENAI_VERSION")
AZURE_OPENAI_GPT_DEPLOYMENT = os.getenv("AZURE_OPENAI_GPT_DEPLOYMENT")
AZURE_OPENAI_GPT_MODEL = os.getenv("AZURE_OPENAI_GPT_MODEL")
AZURE_OPENAI_EMBEDDING_DEPLOYMENT = os.getenv("AZURE_OPENAI_EMBEDDING_DEPLOYMENT")
AZURE_OPENAI_EMBEDDING_MODEL = os.getenv("AZURE_OPENAI_EMBEDDING_MODEL")
AZURE_OPENAI_EMBEDDING_DIMENSIONS = int(os.getenv("AZURE_OPENAI_EMBEDDING_DIMENSIONS", 1536))

# Azure Cognitive Services 
AZURE_AI_MULTISERVICES_ENDPOINT = os.getenv("AZURE_AI_MULTISERVICES_ENDPOINT")
AZURE_AI_MULTISERVICES_KEY = os.getenv("AZURE_AI_MULTISERVICES_KEY")

# Agent Name
AGENT_NAME =  os.getenv("AGENT_NAME")
OPENAI_ANSWER_MODEL =  os.getenv("OPENAI_ANSWER_MODEL")
OPENAI_ANSWER_MODEL_API_VERSION =  os.getenv("OPENAI_ANSWER_MODEL_API_VERSION")

PROJECT_ENDPOINT = "NEED TO CREATE RESOURCE"
MODEL_DEPLOYMENT_NAME = "gpt-4o"

### Agent and Tool Setup

In [ ]:
from functions import functions
functions = FunctionTool(functions=functions)
project_client = AIProjectClient(
    endpoint=PROJECT_ENDPOINT,
    credential=DefaultAzureCredential(),
)

with project_client:
    agents_client = project_client.agents

    # [create agents]
    ten_k_agent_name = "ten_k_bot"
    lead_agent_name = "lead_bot"

    ten_k_agent = agents_client.create_agent(
        model = MODEL_DEPLOYMENT_NAME,
        name=ten_k_agent_name,
        instructions=(
            "Your job is to obtain the loan ammount and then add $10,000 to it. "
        ),
    )

    #some way for the Knowledge Agent to be imported here. This will use the AI Search tool to retreive information from the index on the loan

    #Add connected tools to the lead_bot
    connected_ten_k_agent = ConnectedAgentTool(
        id=ten_k_agent.id, name=ten_k_agent_name, description="Adds $10,000 to the loan amount"
        #add Knowledge Agent here
    )
    

   
    # Create agent with the Connected Agent tool and process assistant run
    agent = agents_client.create_agent(
        model=MODEL_DEPLOYMENT_NAME,
        name=lead_agent_name,
        instructions="You are a helpful assistant, and use the connected agents to perform calculations.",
        tools=[
            connected_ten_k_agent.definitions[0],
        ],
    )
   

    print(f"Created agent, ID: {agent.id}")

  
    # Create thread for communication
    thread = agents_client.threads.create()
    print(f"Created thread, ID: {thread.id}")

    # Create message to thread
    message = agents_client.messages.create(
        thread_id=thread.id,
        role=MessageRole.USER,
        content="Tell me about LN-100000 with John Jones",
    )
    print(f"Created message, ID: {message.id}")

    run = agents_client.runs.create(thread_id=thread.id, agent_id=agent.id)
    print(f"Created run, ID: {run.id}")

    while run.status in ["queued", "in_progress", "requires_action"]:
        time.sleep(1)
        run = agents_client.runs.get(thread_id=thread.id, run_id=run.id)

        if run.status == "requires_action" and isinstance(run.required_action, SubmitToolOutputsAction):
            tool_calls = run.required_action.submit_tool_outputs.tool_calls
            if not tool_calls:
                print("No tool calls provided - cancelling run")
                agents_client.runs.cancel(thread_id=thread.id, run_id=run.id)
                break

            tool_outputs = []
            for tool_call in tool_calls:
                if isinstance(tool_call, RequiredFunctionToolCall):
                    try:
                        print(f"Executing tool call: {tool_call}")
                        output = functions.execute(tool_call)
                        tool_outputs.append(
                            ToolOutput(
                                tool_call_id=tool_call.id,
                                output=output,
                            )
                        )
                    except Exception as e:
                        print(f"Error executing tool_call {tool_call.id}: {e}")

            print(f"Tool outputs: {tool_outputs}")
            if tool_outputs:
                agents_client.runs.submit_tool_outputs(thread_id=thread.id, run_id=run.id, tool_outputs=tool_outputs)

        print(f"Current run status: {run.status}")

    if run.status == "failed":
        print(f"Run failed: {run.last_error}")
  

    for run_step in agents_client.run_steps.list(thread_id=thread.id, run_id=run.id, order=ListSortOrder.ASCENDING):
        if isinstance(run_step.step_details, RunStepToolCallDetails):
            for tool_call in run_step.step_details.tool_calls:
                if isinstance(tool_call, ConnectedAgentTool):
                    print(
                        f"\tAgent: {tool_call.connected_agent.name} " f"query: {tool_call.connected_agent.arguments} ",
                        f"output: {tool_call.connected_agent.output}",
                    )

     # Fetch and log all messages
    messages = agents_client.messages.list(thread_id=thread.id, order=ListSortOrder.ASCENDING)
    for msg in messages:
        if msg.text_messages:
            last_text = msg.text_messages[-1]
            text = last_text.text.value.replace("\u3010", "[").replace("\u3011", "]")
            print(f"{msg.role}: {text}")
    # [END list_messages]

    agents_client.threads.delete(thread.id)
   
